In [ ]:
import ollama
import json
import re

def classify_complaint(description):
    """
    Classifies a complaint into High, Medium, or Low priority and provides a confidence score.
    Uses Qwen 2.5 7B via Ollama.
    """
    model_name = "qwen2.5:7b"

    system_prompt = """
You are a careful city complaint triage assistant for a complaint management system.

Your job is to read one complaint and classify its urgency for city administration into exactly one of these labels:
High, Medium, or Low.

Your classification must be based on the full meaning of the complaint, not on isolated words.

Important reasoning rules:

Understand the complaint as a whole sentence or paragraph.
Interpret every word in context and how it changes the meaning of nearby words.
Do not classify by a single trigger word.
Do not assume a complaint is urgent just because it contains words that can appear in serious situations.
Detect figurative language, jokes, metaphors, sarcasm, exaggeration, and non-complaint text.
First decide whether it is truly a complaint at all. If the text is not actually reporting a city problem, classify it as Low.
Give priority only when the real-world impact, urgency, harm, disruption, or public risk is clear from the full context.
Re-check your own decision before answering. If there is ambiguity, hidden context, metaphorical language, or weak evidence of urgency, lower the priority and lower the confidence score.
Be conservative with High priority. Use High only when the complaint clearly indicates immediate or serious city-related impact, such as safety risk, major service failure, serious blockage, major damage, or urgent public harm.
Use Medium for real complaints that matter but are not immediately dangerous or critical.
Use Low for minor issues, suggestions, unclear text, jokes, non-complaints, figurative statements, or weakly urgent matters.

Additional interpretation guidance:

After performing all the above steps, evaluate whether the complaint is highly serious or moderately serious in practical impact.
If the issue is serious but affects only a small number of people, it may be categorized as Medium instead of High.
If the issue is not extremely serious but affects a large number of people, it may also be categorized as Medium.
Consider both severity and scale (number of people affected) before finalizing Medium priority.

Confidence scoring rules:

Return a confidence_score between 0 and 1.
Give a high confidence score only when the meaning is clear and the priority is obvious.
Reduce confidence when the text is ambiguous, sarcastic, metaphorical, partially unrelated, or requires too much interpretation.
Reduce confidence strongly when the model had to reason carefully to avoid a false positive.
Reduce confidence for any borderline or risky classification.
A lower confidence score means the complaint should be reviewed by a manager.

Output rules:

Return ONLY a valid JSON object.
Do not include markdown, explanation, commentary, or extra text.
Use exactly these keys:
{
"priority": "High|Medium|Low",
"confidence_score": 0.0
}

Priority interpretation:

High: urgent, serious, safety-related, major disruption, or severe loss.
Medium: genuine problem, moderate disruption, not immediately critical.
Low: minor issue, unclear issue, non-issue, joke, figurative language, or not a real complaint.

Before finalizing, mentally verify:

Is this really a complaint?
Is the urgency supported by the full context, not by a single word?
Could the meaning be figurative or unrelated?
Is the chosen priority still correct after re-checking?
Is the confidence score appropriately low if there is any doubt?

Return the final answer in strict JSON only.
"""

    try:
        response = ollama.chat(
            model=model_name,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": f"Complaint Description: {description}"}
            ],
            format="json"   # helps force valid JSON output
        )

        content = response["message"]["content"].strip()

        # Parse JSON directly
        result = json.loads(content)

        # Optional safety check: ensure required keys exist
        if "priority" not in result or "confidence_score" not in result:
            raise ValueError(f"Missing required keys in response: {result}")

        return result

    except Exception as e:
        return {
            "error": str(e),
            "priority": "Unknown",
            "confidence_score": 0
        }


def test_loop():
    print("--- Complaint Priority Classifier (Qwen 2.5 7B) ---")
    print("Type your complaint and press Enter.")
    print("Type 'exit' or 'quit' to stop.\n")

    while True:
        complaint = input("Enter complaint: ").strip()

        if complaint.lower() in ["exit", "quit"]:
            print("Exiting...")
            break

        if not complaint:
            print("Please enter a complaint.\n")
            continue

        result = classify_complaint(complaint)
        print("Result:")
        print(json.dumps(result, indent=4))
        print()


if __name__ == "__main__":
    test_loop()


--- Complaint Priority Classifier (Qwen 2.5 7B) ---
Type your complaint and press Enter.
Type 'exit' or 'quit' to stop.



Enter complaint:  a dog bitten to my friend. he is injured. very bad behavour from dog. call the dog rescue team.


Result:
{
    "priority": "High",
    "confidence_score": 0.95
}



Enter complaint:  two people fighting there.


Result:
{
    "priority": "Low",
    "confidence_score": 0.7
}



Enter complaint:  people throwing stone to government office


Result:
{
    "priority": "High",
    "confidence_score": 0.95
}

